#**L'importance de l'ajustement des hyperparamètres**
Les hyperparamètres sont les réglages que vous définissez avant que l'entraînement ne commence. Contrairement aux "paramètres" que le modèle apprend seul, les hyperparamètres dictent la structure même de l'apprentissage.

Pour l'Arbre de Décision : Si on ne le règle pas, il va créer des branches pour chaque petite faute d'orthographe (Overfitting). Ajuster la profondeur maximale revient à lui dire : "Ne te perds pas dans les détails, concentre-toi sur les mots essentiels".

Pour Naive Bayes : Le paramètre Alpha (Lissage de Laplace) est crucial. Il permet au modèle de ne pas donner une probabilité de 0% à un mot qu'il n'a jamais vu (ce qui ferait échouer toute la classification de la phrase).

In [ ]:
import pandas as pd
import nltk
import random
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from nltk import classify

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# 1. Initialisation des outils
stemmer = SnowballStemmer('french')
stop_words = set(stopwords.words('french'))

# 2. Chargement du dataset (1500 messages)
df = pd.read_csv("/content/messages_support_clients_france.csv", sep=None, engine='python', encoding='utf-8-sig')

def extraire_features(phrase):
    if not isinstance(phrase, str): return {}
    tokens = word_tokenize(phrase.lower(), language='french')
    stems = [stemmer.stem(m) for m in tokens if m.isalpha() and m not in stop_words]
    return {mot: True for mot in stems}

# 3. Préparation du Dataset
dataset_complet = []
for index, row in df.iterrows():
    dataset_complet.append((extraire_features(row['Message']), row['Catégorie']))

random.seed(42)
random.shuffle(dataset_complet)
taille_split = int(len(dataset_complet) * 0.8)
train_set = dataset_complet[:taille_split]
test_set = dataset_complet[taille_split:]

# --- PHASE 3 : AJUSTEMENT DES HYPERPARAMÈTRES ---

# A. Ajustement de l'Arbre de Décision
# entropy_cutoff : contrôle la pureté des nœuds (évite les divisions inutiles)
# support_cutoff : nombre minimum d'exemples pour créer une branche (évite le sur-apprentissage)
dt_tuned = nltk.DecisionTreeClassifier.train(
    train_set,
    entropy_cutoff=0.05,
    support_cutoff=15
)

# B. Optimisation de Naive Bayes
# On entraîne d'abord pour identifier les features inutiles (bruit)
nb_temp = nltk.NaiveBayesClassifier.train(train_set)
top_features = set([f for f, v in nb_temp.most_informative_features(600)])

def extraire_features_tuned(phrase):
    features = extraire_features(phrase)
    # On ne garde que les caractéristiques qui ont un réel impact statistique
    return {f: True for f in features if f in top_features}

train_set_tuned = [(extraire_features_tuned(row['Message']), row['Catégorie']) for index, row in df.iloc[:taille_split].iterrows()]
test_set_tuned = [(extraire_features_tuned(row['Message']), row['Catégorie']) for index, row in df.iloc[taille_split:].iterrows()]

nb_tuned = nltk.NaiveBayesClassifier.train(train_set_tuned)

# 4. Évaluation et Comparaison
acc_nb_tuned = classify.accuracy(nb_tuned, test_set_tuned)
acc_dt_tuned = classify.accuracy(dt_tuned, test_set)

print("\n" + "="*45)
print("RÉSULTATS APRÈS AJUSTEMENT (PHASE 3)")
print("="*45)
print(f"Précision Naive Bayes Ajustée      : {acc_nb_tuned:.2%}")
print(f"Précision Arbre de Décision Ajusté : {acc_dt_tuned:.2%}")
print("="*45)

# Analyse des règles de l'arbre ajusté
print("\nExtrait des décisions de l'Arbre (Pseudocode) :")
print(dt_tuned.pseudocode(depth=2))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



RÉSULTATS APRÈS AJUSTEMENT (PHASE 3)
Précision Naive Bayes Ajustée      : 87.00%
Précision Arbre de Décision Ajusté : 91.33%

Extrait des décisions de l'Arbre (Pseudocode) :
if reçu == None: 
  if factur == None: return 'Aide Technique'
  if factur == True: return 'Facturation'
if reçu == True: return "Problèmes d'expédition"



#**Analyse de l'importance de l'ajustement**
L'ajustement des hyperparamètres a permis de passer d'un apprentissage "par cœur" (parfois bruyant) à une généralisation logique.

Pour l'Arbre de Décision : L'objectif était de simplifier les règles pour ne garder que les mots piliers.

Pour Naive Bayes : L'objectif était de filtrer les termes peu fréquents qui agissaient comme du bruit statistique.

Analyse des résultats techniques

**L'Arbre de Décision**

Le pseudocode généré montre que le modèle a identifié une structure de décision ultra-efficace en seulement trois étapes clés :
Le mot pivot "reçu" : S'il est présent, le modèle classe immédiatement en Expédition.

Le mot pivot "factur" : En l'absence du mot "reçu", si "factur" apparaît, il classe en Facturation.

Si aucun des deux n'est présent, il oriente vers l'Aide Technique.

Conclusion : L'ajustement (support_cutoff=15) a permis de supprimer toutes les branches inutiles liées aux fautes d'orthographe sans perdre un seul point de précision.

**Naive Bayes:**

La baisse à 87,00 % s'explique par le fait que ce modèle a besoin de beaucoup de "petits indices" (plusieurs mots) pour calculer ses probabilités. En limitant le nombre de caractéristiques à 600 pour réduire le bruit, nous avons supprimé certains mots qui, bien que rares, aidaient à affiner la décision.

au final on peut dire que L'Arbre de Décision est désormais le modèle le plus fiable et le plus rapide pour ce type de support client. Ses règles sont claires et faciles à expliquer à un humain.

Pour le déploiement final, nous utiliserons l'Arbre de Décision Ajusté car il offre le meilleur équilibre entre simplicité logique et précision (91,33%).

